# Construcción del panel ambiental 2024 por grilla y semana epidemiológica

1. **Objetivo del notebook**

- Integrar clima, altitud, población, Köppen y NDVI por grilla y semana epidemiológica.

2. **Decisiones metodológicas**

- Excluir filas sin periodo epidemiológico.
- Calcular NDVI por celda-bisemana y aceptar la estadística sólo con al menos 80 % de píxeles válidos.
- Expandir NDVI bisemanal a semanas y mantener la geometría en un archivo separado.

3. **Fuentes**

- `data/raw/environment/era5_weather_altitude_epiweek_2024_population.parquet`.
- `ndvi_file_inventory.csv` y `source_quality_catalog.csv`, generados por `00_source_quality_and_plausibility.ipynb`.
- GeoTIFF NDVI externos registrados en el inventario.

4. **Productos**

- `data/interim/environment/quality_check_environment_grid_week_2024_raw.parquet`.
- `data/interim/environment/environment_grid_week_2024_raw.parquet`.
- `data/processed/environment/grid_geometry.parquet`.


El panel final tiene una fila por `grid_id`, `year` y `epiweek`. Usa el calendario epidemiológico oficial y reutiliza la auditoría NDVI de `00`.

## Configuración y rutas

Se cargan las rutas centralizadas y el calendario epidemiológico definido en `src/epidemiological_calendar.py`.

In [1]:
# Importación de librerías
import os
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from pyproj import CRS
from rasterio.errors import RasterioIOError, WindowError
from rasterio.features import geometry_mask, geometry_window
from shapely import wkb
from IPython.display import display

# Raíz del proyecto
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

# Importación de módulos del proyecto
from src.paths import (
    DATA_DIAGNOSTICS,
    ENVIRONMENT_GRID_WEEK_QUALITY_CHECK_PATH,
    ENVIRONMENT_GRID_WEEK_RAW_PATH,
    GRID_GEOMETRY_PATH,
    RAW_CLIMATE,
)
# Importación de funciones del proyecto
from src.epidemiological_calendar import (
    EPI_YEAR,
    N_EPIWEEKS,
    validate_epidemiological_range,
)

# Configuración de pandas para mostrar todas las columnas y un máximo de 200 filas  
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)


## Carga del Parquet ambiental inicial y validación del CRS

GeoPandas conserva el CRS del GeoParquet. Si falta, se exige `TESIS_GRID_CRS`; no se asume un CRS por defecto.

In [2]:
# Lectura del Parquet ambiental inicial
PARQUET_PATH = RAW_CLIMATE / "era5_weather_altitude_epiweek_2024_population.parquet"
if not PARQUET_PATH.is_file():
    raise FileNotFoundError(f"No existe el Parquet ambiental inicial: {PARQUET_PATH}")

source_crs = None
try:
    source_geo = gpd.read_parquet(PARQUET_PATH)
    source_crs = source_geo.crs
    environment = pd.DataFrame(source_geo)
except Exception as exc:
    environment = pd.read_parquet(PARQUET_PATH)
    print(f"Lectura GeoParquet no disponible ({type(exc).__name__}); se inspeccionará el CRS manualmente.")

# Validación de columnas requeridas
REQUIRED_SOURCE_COLUMNS = {
    "grid_id", "year", "epiweek", "geometry",
    "tmean_c", "tmin_c", "tmax_c",
    "rh_mean", "rh_min", "rh_max", "tp_sum",
    "alt_min", "alt_mean", "alt_max",
    "koppen_fin", "population",
}
missing_source_columns = sorted(REQUIRED_SOURCE_COLUMNS.difference(environment.columns))
if missing_source_columns:
    raise ValueError(
        "El Parquet ambiental no contiene todas las columnas requeridas por "
        f"01_data_aggregation.ipynb: {missing_source_columns}"
    )
print(
    f"Columnas requeridas para la agregación: "
    f"{len(REQUIRED_SOURCE_COLUMNS)}/{len(REQUIRED_SOURCE_COLUMNS)} presentes"
)
# Validación del CRS de la fuente ambiental
configured_crs = os.environ.get("TESIS_GRID_CRS")
if source_crs is None:
    if not configured_crs:
        raise ValueError("La fuente no expone CRS. Define TESIS_GRID_CRS después de verificar el CRS original; no se asumirá EPSG:4326.")
    source_crs = CRS.from_user_input(configured_crs)
    print(f"CRS asignado explícitamente desde TESIS_GRID_CRS={configured_crs}; verificar esta asignación contra la fuente antes de continuar.")
else:
    source_crs = CRS.from_user_input(source_crs)
    if configured_crs and CRS.from_user_input(configured_crs) != source_crs:
        raise ValueError(f"TESIS_GRID_CRS={configured_crs} contradice el CRS de la fuente ({source_crs}).")
# Función para convertir valores a geometría
def as_geometry(value):
    if value is None or pd.isna(value):
        return None
    if hasattr(value, "geom_type"):
        return value
    return wkb.loads(value)

# Conversión de la columna "geometry" a geometría válida
environment = environment.copy()
environment["geometry"] = environment["geometry"].map(as_geometry)
print(f"CRS original validado: {source_crs}")
print(f"Filas cargadas: {len(environment):,}")
display(environment.head())


Columnas requeridas para la agregación: 16/16 presentes
CRS original validado: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum": {"type": "GeodeticReferenceFrame", "name": "World Geodetic System 1984", "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "id": {"authority": "EPSG", "code": 4326}}
Filas cargadas: 177,323


,grid_id,year,epiweek,tmean_c,tmin_c,tmax_c,alt_min,alt_mean,alt_max,rh_mean,rh_min,rh_max,tp_sum,geometry,koppen_fin,population
0,0,2024.0,1.0,21.209301,19.992899,22.276896,25.0,33.596774,43.0,74.356583,71.468781,76.885185,0.016606,MULTIPOLYGON (((-70.36661379399993 -18.3381532...,BWh,0.0
1,0,2024.0,2.0,22.054096,21.573660,22.765982,25.0,33.596774,43.0,73.982315,71.317276,76.284805,0.011030,MULTIPOLYGON (((-70.36661379399993 -18.3381532...,BWh,0.0
2,0,2024.0,3.0,22.568083,22.038534,23.052988,25.0,33.596774,43.0,75.665398,72.660240,78.163437,0.068585,MULTIPOLYGON (((-70.36661379399993 -18.3381532...,BWh,0.0
3,0,2024.0,4.0,23.774378,23.398520,24.111084,25.0,33.596774,43.0,72.535446,70.525070,74.601913,0.008243,MULTIPOLYGON (((-70.36661379399993 -18.3381532...,BWh,0.0
4,0,2024.0,5.0,22.861515,22.499634,23.158884,25.0,33.596774,43.0,73.460915,72.232567,74.605240,0.005395,MULTIPOLYGON (((-70.36661379399993 -18.3381532...,BWh,0.0


## Exclusión de las filas sin periodo

Se eliminan las filas sin `year` ni `epiweek` antes de integrar NDVI. Un periodo parcialmente informado detiene la ejecución.

In [3]:
PERIOD_COLUMNS = ["year", "epiweek"]
EXPECTED_ROWS_WITHOUT_PERIOD = 55

# Normalización estricta y conteo único de campos temporales ausentes por fila
environment[PERIOD_COLUMNS] = (
    environment[PERIOD_COLUMNS]
    .apply(pd.to_numeric, errors="raise")
    .astype("Int64")
)
missing_period_fields = environment[PERIOD_COLUMNS].isna().sum(axis=1)

# Un periodo parcialmente informado es ambiguo y detiene el procesamiento
partial_period_mask = missing_period_fields.eq(1)
if partial_period_mask.any():
    examples = environment.loc[partial_period_mask, ["grid_id", *PERIOD_COLUMNS]].head(10).to_dict("records")
    raise ValueError(
        "Existen filas donde falta solo year o solo epiweek; "
        f"ejemplos: {examples}"
    )

# Identificación y exclusión de las filas donde faltan year y epiweek
missing_period_mask = missing_period_fields.eq(len(PERIOD_COLUMNS))
excluded_row_count = int(missing_period_mask.sum())
if excluded_row_count != EXPECTED_ROWS_WITHOUT_PERIOD:
    raise ValueError(
        f"Se esperaban {EXPECTED_ROWS_WITHOUT_PERIOD} filas sin periodo; "
        f"se encontraron {excluded_row_count}. Revisar el archivo antes de continuar."
    )
excluded_grid = environment.loc[missing_period_mask, "grid_id"]
if excluded_grid.isna().any():
    raise ValueError("Las filas excluidas deben conservar grid_id para informar su conteo.")

temporal = environment.loc[missing_period_fields.eq(0)].copy()
if temporal["grid_id"].isna().any():
    raise ValueError("Después de excluir las filas sin periodo quedan grid_id nulos.")
validate_epidemiological_range(temporal)
excluded_grid_count = int(excluded_grid.nunique())
print(f"Filas excluidas por falta simultánea de year y epiweek: {excluded_row_count:,}")
print(f"grid_id afectados por la exclusión: {excluded_grid_count:,}")
print(f"Filas temporales restantes: {len(temporal):,}")


Filas excluidas por falta simultánea de year y epiweek: 55
grid_id afectados por la exclusión: 55
Filas temporales restantes: 177,268


## Geometría canónica del universo temporal

Se guarda una geometría única por cada `grid_id` que permanece en el panel.

In [4]:
# Validación de llaves duplicadas en el universo temporal
KEY_COLUMNS = ["grid_id", "year", "epiweek"]
if temporal.duplicated(KEY_COLUMNS).any():
    examples = temporal.loc[temporal.duplicated(KEY_COLUMNS, keep=False), KEY_COLUMNS].head(10).to_dict("records")
    raise ValueError(f"La fuente ya contiene llaves grid_id-year-epiweek duplicadas: {examples}")

# Validación de geometrías en el universo temporal
spatial = temporal[["grid_id", "geometry"]].copy()
if spatial["geometry"].isna().any() or spatial["geometry"].map(lambda geom: geom.is_empty).any():
    raise ValueError("Existen geometrías nulas o vacías en el universo temporal.")
if spatial["geometry"].map(lambda geom: not geom.is_valid).any():
    raise ValueError("Existen geometrías inválidas en el universo temporal.")
spatial["_geometry_key"] = spatial["geometry"].map(lambda geom: geom.wkb_hex)
geometry_counts = spatial.groupby("grid_id")["_geometry_key"].nunique()
if geometry_counts.ne(1).any():
    raise ValueError("Cada grid_id debe tener una sola geometría.")

# Creación del GeoDataFrame de geometría canónica
grid_geometry = gpd.GeoDataFrame(
    spatial.drop_duplicates("grid_id")[["grid_id", "geometry"]],
    geometry="geometry",
    crs=source_crs,
).sort_values("grid_id").reset_index(drop=True)
if grid_geometry["grid_id"].isna().any() or grid_geometry["grid_id"].duplicated().any():
    raise ValueError("La geometría canónica tiene grid_id nulos o duplicados.")
if set(grid_geometry["grid_id"]) != set(temporal["grid_id"]):
    raise ValueError("El universo espacial no coincide con los grid_id temporales.")
print(f"grid_id en el universo temporal y geométrico: {len(grid_geometry):,}")
print(f"CRS de grid_geometry: {grid_geometry.crs}")
display(grid_geometry.head())


grid_id en el universo temporal y geométrico: 3,409
CRS de grid_geometry: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum": {"type": "GeodeticReferenceFrame", "name": "World Geodetic System 1984", "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "id": {"authority": "EPSG", "code": 4326}}


,grid_id,geometry
0,0,"MULTIPOLYGON (((-70.36661 -18.33815, -70.36389..."
1,1,"MULTIPOLYGON (((-70.37086 -18.35941, -70.37096..."
2,2,"MULTIPOLYGON (((-70.35901 -18.37171, -70.35907..."
3,4,"MULTIPOLYGON (((-70.33996 -18.31873, -70.33767..."
4,5,"MULTIPOLYGON (((-70.35654 -18.32905, -70.35356..."


## Insumos y cobertura NDVI

- Reutiliza el inventario y el catálogo de calidad de `00`.
- Organiza los GeoTIFF por métrica y bisemana para la agregación zonal.
- **Exige al menos 80 % de píxeles válidos por celda; bajo ese umbral, la estadística queda nula.**
- Conserva conteos y porcentajes de cobertura para `02`.

In [5]:
# El año epidemiológico de 52 semanas se representa mediante 26 productos bisemanales
N_BIWEEKS = N_EPIWEEKS // 2

# Una estadística zonal se acepta solamente con al menos 80 % de píxeles válidos
MIN_NDVI_COVERAGE_PCT = 80.0

# Relación entre cada bisemana NDVI y las dos semanas epidemiológicas que representa
BIWEEK_TO_EPIWEEKS = {biweek: (2 * biweek - 1, 2 * biweek) for biweek in range(1, N_BIWEEKS + 1)}
# Control mínimo para confirmar que la relación cubre desde SE1 hasta SE52
if BIWEEK_TO_EPIWEEKS[1] != (1, 2) or BIWEEK_TO_EPIWEEKS[26] != (51, 52):
    raise ValueError("La relación BI↔SE no cubre exactamente SE1..SE52.")

# Estadística espacial aplicada a los píxeles de cada GeoTIFF dentro de cada celda
# La desviación estándar ya viene calculada en el raster; aquí se promedia dentro de la celda
STAT_CONFIG = {
    "ndvi_media_bisemanal": {"aggregation": "mean"},
    "ndvi_maximo_bisemanal": {"aggregation": "max"},
    "ndvi_minimo_bisemanal": {"aggregation": "min"},
    "ndvi_desvest_bisemanal": {"aggregation": "mean"},
}
NDVI_COLUMNS = list(STAT_CONFIG)

# Rutas de los dos productos de 00 que necesita la agregación
NDVI_AUDIT_DIR = DATA_DIAGNOSTICS / "source_exploration"
NDVI_INVENTORY_PATH = NDVI_AUDIT_DIR / "ndvi_file_inventory.csv"
NDVI_QUALITY_CATALOG_PATH = NDVI_AUDIT_DIR / "source_quality_catalog.csv"
# Estos CSV deben existir porque 01 no vuelve a inventariar ni auditar los GeoTIFF
required_audit_files = [
    NDVI_INVENTORY_PATH,
    NDVI_QUALITY_CATALOG_PATH,
]
missing_audit_files = [path for path in required_audit_files if not path.is_file()]
if missing_audit_files:
    raise FileNotFoundError(
        "Faltan diagnósticos NDVI. Ejecuta completo "
        f"00_source_quality_and_plausibility.ipynb: {missing_audit_files}"
    )

# El inventario aporta las rutas; el catálogo aporta el resultado consolidado de calidad
ndvi_inventory = pd.read_csv(NDVI_INVENTORY_PATH)
source_quality_catalog = pd.read_csv(NDVI_QUALITY_CATALOG_PATH)

# Se comprueba solo la estructura mínima necesaria para interpretar el catálogo
required_catalog_columns = {"source", "control", "state", "evidence", "action"}
if required_catalog_columns.difference(source_quality_catalog.columns):
    raise ValueError("El catálogo de calidad no tiene el esquema esperado.")
# Se aíslan los controles correspondientes a NDVI
ndvi_quality_catalog = source_quality_catalog.loc[source_quality_catalog["source"].eq("NDVI")].copy()
required_ndvi_controls = {
    "inventario BI01–BI26",
    "alineación espacial por bisemana",
    "rangos físicos de píxel",
    "mínimo ≤ media ≤ máximo por píxel",
}
# Los controles deben estar documentados en el catálogo de 00
missing_ndvi_controls = required_ndvi_controls.difference(ndvi_quality_catalog["control"])
if missing_ndvi_controls:
    raise ValueError(
        "La auditoría NDVI está incompleta; faltan controles "
        f"{sorted(missing_ndvi_controls)}. Ejecuta el escaneo global del notebook 00."
    )
# Un estado crítico informado por 00 impide comenzar la agregación
critical_ndvi = ndvi_quality_catalog.loc[ndvi_quality_catalog["state"].eq("crítico")]
if not critical_ndvi.empty:
    raise ValueError(
        "La auditoría NDVI contiene controles críticos: "
        f"{critical_ndvi[['control', 'evidence']].to_dict('records')}"
    )

# El inventario solo necesita identificar la métrica, la bisemana y la ruta
required_inventory_columns = {"metric", "biweek", "path"}
missing_inventory_columns = required_inventory_columns.difference(ndvi_inventory.columns)
if missing_inventory_columns:
    raise ValueError(
        "El inventario NDVI está desactualizado; faltan columnas "
        f"{sorted(missing_inventory_columns)}. Vuelve a ejecutar el notebook 00."
    )
# La bisemana se normaliza como entero para usarla como clave del diccionario
ndvi_inventory["biweek"] = pd.to_numeric(ndvi_inventory["biweek"], errors="raise").astype(int)

# Estructura resultante: {métrica: {bisemana: ruta_del_GeoTIFF}}
ndvi_files_by_metric = {
    column: {
        int(row.biweek): Path(row.path)
        for row in ndvi_inventory.loc[
            ndvi_inventory["metric"].eq(column)
        ].itertuples(index=False)
    }
    for column in NDVI_COLUMNS
}


In [6]:
# Calcula una estadística NDVI para cada polígono de la grilla usando un GeoTIFF
# y conserva los conteos necesarios para evaluar la cobertura de píxeles.
def zonal_stat_for_raster(grid: gpd.GeoDataFrame, raster_path: Path, value_column: str, aggregation: str) -> pd.DataFrame:
    # Relación entre el nombre configurado y la operación aplicada a los píxeles válidos
    aggregations = {"mean": lambda values: values.mean(), "max": lambda values: values.max(), "min": lambda values: values.min()}
    if aggregation not in aggregations:
        raise ValueError(f"Agregación no soportada: {aggregation}")
    try:
        # El raster se abre una sola vez mientras se recorren todas las celdas
        with rasterio.open(raster_path) as src:
            # El CRS es necesario para llevar los polígonos al sistema del raster
            if src.crs is None:
                raise ValueError(f"El GeoTIFF no tiene CRS: {raster_path}")
            raster_grid = grid.to_crs(src.crs) if grid.crs != src.crs else grid
            rows = []
            # Cada iteración produce una observación para un grid_id
            for grid_id, geom in zip(raster_grid["grid_id"], raster_grid.geometry):
                try:
                    # Se lee solo la ventana rectangular que rodea al polígono
                    window = geometry_window(src, [geom], pad_x=0, pad_y=0)
                    data = src.read(1, window=window, masked=True)
                except WindowError:
                    # La celda no intersecta la extensión del raster
                    rows.append((grid_id, np.nan, 0, 0, np.nan))
                    continue
                if data.size == 0:
                    # La intersección no contiene una matriz de píxeles procesable
                    rows.append((grid_id, np.nan, 0, 0, np.nan))
                    continue
                # La máscara conserva los píxeles tocados por el polígono de la celda
                transform = src.window_transform(window)
                mask = geometry_mask([geom], out_shape=data.shape, transform=transform, invert=True, all_touched=True)
                # Denominador de cobertura: todos los píxeles ubicados dentro de la celda
                total_pixel_count = int(mask.sum())
                # Se excluyen simultáneamente los píxeles fuera del polígono y los ya enmascarados
                values = np.ma.array(data, mask=np.ma.getmaskarray(data) | ~mask)
                # También se excluyen el valor nodata declarado y los valores no finitos
                if src.nodata is not None:
                    values = np.ma.masked_equal(values, src.nodata)
                values = np.ma.masked_invalid(values)
                # Numerador de cobertura: píxeles que conservan un valor NDVI válido
                pixel_count = int(values.count())
                coverage_pct = 100.0 * pixel_count / total_pixel_count if total_pixel_count else np.nan
                # El valor zonal solo se calcula cuando la cobertura alcanza el 80 %
                has_sufficient_coverage = (
                    pixel_count > 0
                    and np.isfinite(coverage_pct)
                    and coverage_pct >= MIN_NDVI_COVERAGE_PCT
                )
                value = (
                    float(aggregations[aggregation](values))
                    if has_sufficient_coverage
                    else np.nan
                )
                # Aunque el valor quede nulo, los conteos y la cobertura se conservan
                rows.append((grid_id, value, pixel_count, total_pixel_count, coverage_pct))
            # Una fila por celda, con el valor agregado y su evidencia de cobertura
            return pd.DataFrame(rows, columns=["grid_id", value_column, f"{value_column}_pixel_count", f"{value_column}_total_pixel_count", f"{value_column}_coverage_pct"])
    except RasterioIOError as exc:
        # Se agrega la ruta al error original para identificar el archivo problemático
        raise RasterioIOError(f"No se pudo abrir el GeoTIFF: {raster_path}") from exc


In [7]:
# Convierte una serie de 26 valores bisemanales por celda en 52 valores semanales
def expand_biweeks_to_epiweeks(frame: pd.DataFrame, value_column: str) -> pd.DataFrame:
    # El orden por celda y bisemana permite relacionar cada periodo con el siguiente
    frame = frame.sort_values(["grid_id", "biweek"]).copy()
    # Estas columnas documentan cuántos píxeles respaldan cada valor NDVI
    count_columns = [f"{value_column}_pixel_count", f"{value_column}_total_pixel_count", f"{value_column}_coverage_pct"]

    # Se desplaza una copia para unir BI actual con BI siguiente dentro de cada grid_id
    next_frame = frame[["grid_id", "biweek", value_column, *count_columns]].copy()
    next_frame["biweek"] = next_frame["biweek"] - 1
    next_frame = next_frame.rename(columns={value_column: f"next_{value_column}", **{c: f"next_{c}" for c in count_columns}})
    joined = frame.merge(next_frame, on=["grid_id", "biweek"], how="left", validate="one_to_one")

    # La primera semana de cada par conserva directamente el valor de su bisemana
    first = frame.copy()
    # La segunda semana se construye combinando la bisemana actual con la siguiente
    second = frame.copy()
    first["epiweek"] = [BIWEEK_TO_EPIWEEKS[int(b)][0] for b in first["biweek"]]
    second["epiweek"] = [BIWEEK_TO_EPIWEEKS[int(b)][1] for b in second["biweek"]]

    # Para la segunda semana se exige que ambos valores existan; si falta uno, queda nulo
    second[value_column] = joined[[value_column, f"next_{value_column}"]].mean(axis=1, skipna=False).to_numpy()
    # BI26 no tiene una bisemana posterior: SE52 conserva el valor de BI26
    last_biweek = second["biweek"].eq(N_BIWEEKS)
    second.loc[last_biweek, value_column] = joined.loc[last_biweek, value_column].to_numpy()

    # La evidencia de cobertura de la segunda semana adopta el valor más conservador
    for column in count_columns:
        second[column] = joined[[column, f"next_{column}"]].min(axis=1, skipna=True).to_numpy()

    # Se reúnen las semanas impares y pares y se incorpora el año epidemiológico
    weekly = pd.concat([first, second], ignore_index=True)
    weekly["year"] = EPI_YEAR
    return weekly.sort_values(["grid_id", "epiweek"]).reset_index(drop=True)

# Calcula todas las métricas y las reúne en un único panel NDVI semanal
def build_ndvi_epiweek_dataframe(grid: gpd.GeoDataFrame) -> pd.DataFrame:
    stat_frames = []
    # Cada métrica utiliza su propia operación espacial definida en STAT_CONFIG
    for column, config in STAT_CONFIG.items():
        files = ndvi_files_by_metric[column]
        biweekly_parts = []
        # Se calcula una estadística zonal para cada GeoTIFF bisemanal
        for biweek, raster_path in sorted(files.items()):
            print(f"Calculando {column} BI{biweek:02d}: {raster_path.name}")
            zonal = zonal_stat_for_raster(grid, raster_path, column, config["aggregation"])
            zonal["biweek"] = biweek
            biweekly_parts.append(zonal)
        # Se apilan las bisemanas de la métrica y después se expanden a 52 semanas
        stat_frames.append(expand_biweeks_to_epiweeks(pd.concat(biweekly_parts, ignore_index=True), column))

    # Las cuatro métricas se unen por celda, año, semana y bisemana
    result = stat_frames[0]
    for frame in stat_frames[1:]:
        result = result.merge(frame, on=["grid_id", "year", "epiweek", "biweek"], how="outer", validate="one_to_one")
    return result

# Ejecución de la agregación NDVI sobre la geometría canónica del panel
ndvi_week = build_ndvi_epiweek_dataframe(grid_geometry)
# El resultado debe contener exactamente 52 semanas por cada grid_id
if len(ndvi_week) != len(grid_geometry) * N_EPIWEEKS:
    raise ValueError("La expansión NDVI no produjo exactamente 52 semanas por grid_id.")
# La combinación espacial y temporal debe ser única
if ndvi_week.duplicated(["grid_id", "year", "epiweek"]).any():
    raise ValueError("La expansión NDVI produjo llaves duplicadas.")


Calculando ndvi_media_bisemanal BI01: ndvi_arica_y_parinacota_2024_BI01_mean.tif
Calculando ndvi_media_bisemanal BI02: ndvi_arica_y_parinacota_2024_BI02_mean.tif
Calculando ndvi_media_bisemanal BI03: ndvi_arica_y_parinacota_2024_BI03_mean.tif
Calculando ndvi_media_bisemanal BI04: ndvi_arica_y_parinacota_2024_BI04_mean.tif
Calculando ndvi_media_bisemanal BI05: ndvi_arica_y_parinacota_2024_BI05_mean.tif
Calculando ndvi_media_bisemanal BI06: ndvi_arica_y_parinacota_2024_BI06_mean.tif
Calculando ndvi_media_bisemanal BI07: ndvi_arica_y_parinacota_2024_BI07_mean.tif
Calculando ndvi_media_bisemanal BI08: ndvi_arica_y_parinacota_2024_BI08_mean.tif
Calculando ndvi_media_bisemanal BI09: ndvi_arica_y_parinacota_2024_BI09_mean.tif
Calculando ndvi_media_bisemanal BI10: ndvi_arica_y_parinacota_2024_BI10_mean.tif
Calculando ndvi_media_bisemanal BI11: ndvi_arica_y_parinacota_2024_BI11_mean.tif
Calculando ndvi_media_bisemanal BI12: ndvi_arica_y_parinacota_2024_BI12_mean.tif
Calculando ndvi_media_bisema

## Integración y validaciones finales

Se integran ERA5 y NDVI. Se guarda un panel de control con cobertura, otro reducido para limpieza y la geometría por separado.

In [8]:
# Se eliminan posibles columnas NDVI antiguas para no mezclarlas con el cálculo actual
# y luego se une ERA5 (`temporal`) con el panel semanal de NDVI (`ndvi_week`).
panel = (
    temporal.drop(
        columns=[
            column
            for column in temporal.columns
            if column.startswith("ndvi_") or column == "biweek"
        ],
        errors="ignore",
    ).merge(
        ndvi_week,
        on=["grid_id", "year", "epiweek"],
        how="left",          # Conserva todas las filas válidas de ERA5
        validate="one_to_one", # Exige una fila única por celda y semana en ambos DataFrames
    )
)

# El producto final tiene unidad espacial de grilla, por lo que se retiran columnas distritales
district_columns = [column for column in panel.columns if any(token in column.casefold() for token in ("distrito", "district"))]
if district_columns:
    panel = panel.drop(columns=district_columns)
    print(f"Columnas distritales excluidas del panel: {district_columns}")

# La geometría se guarda por separado para no repetir el mismo polígono en las 52 semanas
panel_raw = panel.drop(columns=["geometry"], errors="ignore").copy()
# El panel tabular y el archivo geográfico deben representar exactamente los mismos grid_id
if set(panel_raw["grid_id"]) != set(grid_geometry["grid_id"]):
    raise ValueError("Los grid_id del panel y de grid_geometry no coinciden.")

# Cada combinación celda–año–semana debe aparecer una sola vez
if panel_raw.duplicated(["grid_id", "year", "epiweek"]).any():
    raise ValueError("La llave grid_id-year-epiweek no es única.")

# Se confirma que todas las etiquetas temporales correspondan a SE1..SE52 de 2024
validate_epidemiological_range(panel_raw)

# Cada grid_id debe aportar exactamente una fila por cada una de las 52 semanas
rows_per_grid = panel_raw.groupby("grid_id").size()
if not rows_per_grid.eq(N_EPIWEEKS).all():
    raise ValueError(f"No hay exactamente 52 filas por grid_id: {rows_per_grid[~rows_per_grid.eq(N_EPIWEEKS)].to_dict()}")

# Las filas conservadas más las 55 excluidas deben reconstruir el tamaño de la fuente
if len(panel_raw) + excluded_row_count != len(environment):
    raise ValueError("Las filas excluidas no están contabilizadas exactamente una vez.")

# Ninguna fila del producto final puede quedar sin año o semana
if panel_raw[["year", "epiweek"]].isna().any().any():
    raise ValueError("El panel final contiene llaves temporales nulas.")

# Se comprueba que las cuatro métricas NDVI y sus variables de cobertura estén presentes
for column in NDVI_COLUMNS:
    # La métrica debe existir y contener al menos algún valor no nulo
    if column not in panel_raw or panel_raw[column].isna().all():
        raise ValueError(f"Falta cobertura de valores para la métrica NDVI: {column}")
    # Cada valor NDVI debe estar acompañado por conteos y porcentaje de cobertura
    for suffix in ("pixel_count", "total_pixel_count", "coverage_pct"):
        if f"{column}_{suffix}" not in panel_raw.columns:
            raise ValueError(f"Falta columna de cobertura NDVI: {column}_{suffix}")

# El producto de control conserva la trazabilidad usada por 02
QUALITY_CHECK_ONLY_COLUMNS = [
    "biweek",
    *[
        f"{column}_{suffix}"
        for column in NDVI_COLUMNS
        for suffix in ("pixel_count", "total_pixel_count", "coverage_pct")
    ],
]
panel_quality_check = panel_raw.copy()
# El panel raw destinado a limpieza, EDA y modelamiento excluye columnas diagnósticas
panel_raw = panel_quality_check.drop(columns=QUALITY_CHECK_ONLY_COLUMNS)

# Resumen final antes de guardar los productos
print(f"Filas excluidas definitivamente: {excluded_row_count:,}")
print(f"Panel de control de calidad: {len(panel_quality_check):,} filas × {panel_quality_check.shape[1]} columnas")
print(f"Panel raw reducido: {len(panel_raw):,} filas × {panel_raw.shape[1]} columnas")
display(panel_raw.head())


Filas excluidas definitivamente: 55
Panel de control de calidad: 177,268 filas × 32 columnas
Panel raw reducido: 177,268 filas × 19 columnas


,grid_id,year,epiweek,tmean_c,tmin_c,tmax_c,alt_min,alt_mean,alt_max,rh_mean,rh_min,rh_max,tp_sum,koppen_fin,population,ndvi_media_bisemanal,ndvi_maximo_bisemanal,ndvi_minimo_bisemanal,ndvi_desvest_bisemanal
0,0,2024,1,21.209301,19.992899,22.276896,25.0,33.596774,43.0,74.356583,71.468781,76.885185,0.016606,BWh,0.0,0.050019,0.081831,-0.021624,0.004055
1,0,2024,2,22.054096,21.573660,22.765982,25.0,33.596774,43.0,73.982315,71.317276,76.284805,0.011030,BWh,0.0,0.048979,0.080882,-0.004966,0.003381
2,0,2024,3,22.568083,22.038534,23.052988,25.0,33.596774,43.0,75.665398,72.660240,78.163437,0.068585,BWh,0.0,0.047940,0.079934,0.011691,0.002707
3,0,2024,4,23.774378,23.398520,24.111084,25.0,33.596774,43.0,72.535446,70.525070,74.601913,0.008243,BWh,0.0,0.043426,0.070744,0.004822,0.001353
4,0,2024,5,22.861515,22.499634,23.158884,25.0,33.596774,43.0,73.460915,72.232567,74.605240,0.005395,BWh,0.0,0.038912,0.061555,-0.002047,0.000000


In [9]:
# Se crean las carpetas de destino si todavía no existen
ENVIRONMENT_GRID_WEEK_RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
ENVIRONMENT_GRID_WEEK_QUALITY_CHECK_PATH.parent.mkdir(parents=True, exist_ok=True)
GRID_GEOMETRY_PATH.parent.mkdir(parents=True, exist_ok=True)

# Panel completo para auditoría de cobertura NDVI en 02
panel_quality_check.to_parquet(ENVIRONMENT_GRID_WEEK_QUALITY_CHECK_PATH, index=False)
# Panel semanal reducido para limpieza, EDA y modelamiento
panel_raw.to_parquet(ENVIRONMENT_GRID_WEEK_RAW_PATH, index=False)
# Geometría canónica: un único polígono por grid_id
grid_geometry.to_parquet(GRID_GEOMETRY_PATH, index=False)

# Se informan las rutas efectivas de los productos generados
print(f"Panel de control de calidad guardado en: {ENVIRONMENT_GRID_WEEK_QUALITY_CHECK_PATH}")
print(f"Panel ambiental raw reducido guardado en: {ENVIRONMENT_GRID_WEEK_RAW_PATH}")
print(f"Geometría canónica guardada en: {GRID_GEOMETRY_PATH}")


Panel de control de calidad guardado en: C:\tesis_aedes_grilla_2024\data\interim\environment\quality_check_environment_grid_week_2024_raw.parquet
Panel ambiental raw reducido guardado en: C:\tesis_aedes_grilla_2024\data\interim\environment\environment_grid_week_2024_raw.parquet
Geometría canónica guardada en: C:\tesis_aedes_grilla_2024\data\processed\environment\grid_geometry.parquet


## Resultado esperado

Se generan el panel de control, el panel raw reducido y la geometría canónica. Los GeoTIFF permanecen en su ubicación externa.